In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,2,3,4,5,6,7"  # Exclude GPU 1 (in use)

import json
import glob
from pathlib import Path

import torch
print(f"CUDA available? {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
from vllm import LLM, SamplingParams
from transformers import AutoProcessor

MODEL = "google/gemma-4-31B-it"

processor = AutoProcessor.from_pretrained(MODEL)
llm = LLM(
    model=MODEL,
    tensor_parallel_size=2,
    max_model_len=8192,
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
)

print(f"Model loaded: {MODEL}")

In [ ]:
SYSTEM_PROMPT = """You are an expert at reading and summarizing academic geography papers.

Given the full text of a paper in markdown, write a methodology abstract: a concise paragraph (150-250 words) summarizing the paper's methodological approach.

Important guidelines:
- Match the voice and style of the original paper. If the authors write formally, write formally. If they use first person ("we conducted"), mirror that. Capture their disciplinary vocabulary and tone.
- Write flowing academic prose—no bullet points, headers, or labels like "Study type:" or "Methods:"
- Cover the study type, methods, data sources, spatial scale, study area, time period, and key variables as relevant, woven naturally into the prose
- Omit any details not mentioned in the paper

The output should read as if the original authors wrote a methods-focused abstract for their own paper.
"""

print("System prompt ready.")

System prompt ready.


In [ ]:
MARKDOWN_DIR = "/gpfs1/home/j/s/jstonge1/rural-geog-classif/parse/output/docling"
md_files = sorted(glob.glob(os.path.join(MARKDOWN_DIR, "*.md")))
print(f"Found {len(md_files)} markdown papers")

# Show first few
for f in md_files[:5]:
    print(f"  {Path(f).name}")

In [ ]:
def summarize_paper(paper_text, max_input_chars=24000):
    """Summarize methodology of a single paper."""
    # Truncate very long papers to fit context window
    if len(paper_text) > max_input_chars:
        paper_text = paper_text[:max_input_chars] + "\n\n[... truncated ...]"
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Here is the paper:\n\n{paper_text}"},
    ]
    
    prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    outputs = llm.generate(
        [{"prompt": prompt}],
        sampling_params=SamplingParams(temperature=0.0, max_tokens=1024),
    )
    
    return outputs[0].outputs[0].text.strip()

print("summarize_paper() ready.")

summarize_paper() ready.


## Test on a single paper

In [ ]:
import textwrap

# Test with one paper
test_file = [f for f in md_files if "10.1080_00045608.2014.892338" in f][0]
with open(test_file) as f:
    paper_text = f.read()

print(f"Paper: {Path(test_file).name}")
print(f"Length: {len(paper_text)} chars")
print("---")
result = summarize_paper(paper_text)
print(textwrap.fill(result, width=80))

Paper: 10.1080_00045608.2014.892338.md
Length: 113364 chars
---


Rendering prompts:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%| | 0/1 [00:00<?, ?it

To examine how the level of detail in market representation influences projected
land-use patterns, we employed an abstract, agent-based model (ABM) titled Land
Use in an eXurban Environment (LUXE). Designed to generalize across existing
spatially explicit models of the North American urban-rural fringe, LUXE
utilizes a simplified environment of homogeneous land parcels where price
variation is driven by locational characteristics—specifically proximity to the
urban center and local open-space amenities—as well as the heterogeneous
preferences and budgets of agents. We conducted a series of experiments
incorporating agents representing land buyers and sellers to scrutinize the
effects of specific market-based decision-making behaviors. Specifically, we
tested the impacts of incorporating budget constraints and competitive bidding
mechanisms against simpler utility maximization algorithms. By varying the
complexity of these market interactions, we analyzed the resulting quantity of
land

## Run on all papers

In [ ]:
import time
from tqdm import tqdm

OUTPUT_DIR = "/gpfs1/home/j/s/jstonge1/rural-geog-classif/summarize/output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = {}
errors = []

for md_path in tqdm(md_files, desc="Summarizing"):
    doi = Path(md_path).stem
    
    try:
        with open(md_path) as f:
            paper_text = f.read()
        
        summary = summarize_paper(paper_text)
        results[doi] = summary
        
        # Save individual summary
        out_path = os.path.join(OUTPUT_DIR, f"{doi}.md")
        with open(out_path, "w") as f:
            f.write(f"# {doi}\n\n{summary}\n")
    
    except Exception as e:
        errors.append((doi, str(e)))
        print(f"Error on {doi}: {e}")

print(f"\nDone: {len(results)} summaries, {len(errors)} errors")

In [ ]:
# Also save all results as a single JSON for easy loading
with open(os.path.join(OUTPUT_DIR, "all_summaries_same_style.json"), "w") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {OUTPUT_DIR}/all_summaries_same_style.json")

Saved to /gpfs1/home/j/s/jstonge1/rural-geog-classif/summarize/output/all_summaries_same_style.json
